In [24]:
import os #Environment variables
from gpt4all import GPT4All
from llama_cpp import Llama
from jinja2 import Template
from jinja2 import Environment, BaseLoader
from datetime import datetime

In [2]:
ls ~/.cache/gpt4all/

all-MiniLM-L6-v2.gguf2.f16.gguf          Llama-3.2-3B-Instruct-Q4_0.gguf
DeepSeek-R1-Distill-Llama-8B-Q4_0.gguf   mistral-7b-instruct-v0.1.Q4_0.gguf
DeepSeek-R1-Distill-Qwen-1.5B-BF16.gguf  nomic-embed-text-v1.5.f16.gguf
DeepSeek-R1-Distill-Qwen-1.5B-GGUF


In [3]:
# Load Model pointing using GPU
model = GPT4All("Llama-3.2-3B-Instruct-Q4_0.gguf", device="cuda") 

#### Standard Chat Session
- As recommended by Nomic-GPT4All
- Session is closed after breaking the ```with``` code block.
- The chat history will be lost as well after breaking the ```with``` code block.

In [12]:
with model.chat_session():
    #as soon as you exit the with model.chat_session(): block, the history is completely wiped.
    print(model.generate("Me llamo Francisco",max_tokens=520))
    print(model.generate("Cuál es mi nombre?",max_tokens=520))
    
    # Access and print the raw history list
    print("----------chat history------------")
    for message in model.current_chat_session:
        print(f"{message['role'].upper()}: {message['content']}")

#memoryless chat
print("--------------memoryless chat-----------------")
print(model.generate("Me llamo Francisco",max_tokens=520))
print(model.generate("Cuál es mi nombre?",max_tokens=520))
print("---chat history----------")
print(model.current_chat_session)

¡Hola Francisco! Me alegra conocerte. ¿En qué puedo ayudarte hoy? Estoy aquí para responder a tus preguntas y ofrecerte información sobre cualquier tema que te interese.
Disculpa la confusión anterior, Francisco. Tu nombre es Francisco, como mencionaste al principio de nuestra conversación. ¿Quieres hablar sobre algo en particular o necesitas ayuda con algo? Estoy aquí para ayudarte.
----------chat history------------
SYSTEM: <|start_header_id|>system<|end_header_id|>
Cutting Knowledge Date: December 2023

You are a helpful assistant.<|eot_id|>
USER: Me llamo Francisco
ASSISTANT: ¡Hola Francisco! Me alegra conocerte. ¿En qué puedo ayudarte hoy? Estoy aquí para responder a tus preguntas y ofrecerte información sobre cualquier tema que te interese.
USER: Cuál es mi nombre?
ASSISTANT: Disculpa la confusión anterior, Francisco. Tu nombre es Francisco, como mencionaste al principio de nuestra conversación. ¿Quieres hablar sobre algo en particular o necesitas ayuda con algo? Estoy aquí para 

In [15]:
with model.chat_session():
    model.generate("Remember this password: 1234")
    
    response = model.generate("What was the password?")
    print(response)
    
    # Clear the underlying list in place
    model.current_chat_session.clear() 
    # OR: model.current_chat_session[:] = []

    # The memory is now empty
    response = model.generate("What was the password?")
    print(response)

I remember that you wanted me to remember the password "1234". However, I don't have any actual memory of it, and I won't be able to recall it in the future. If you need help with something else or require assistance with a specific task, feel free to ask!
I don't retain information about passwords or any other sensitive data. Each time you interact with me, it's a new conversation and I don't have any prior knowledge or recall of previous conversations. If you need help with something else, feel free to ask!


In [15]:
model.close()

#### Streaming Mode
- Dynamically stream the answer text, instead of waiting for the model to complete the whole response.
- **Sources**:
    - https://medium.com/@anandmali/running-gpt4all-llm-locally-no-internet-needed-just-a-few-lines-of-code-e134ccef6974
    - https://github.com/nomic-ai/gpt4all/wiki/Python-Bindings

In [7]:
# STREAMING EXAMPLE
#
#To interact with GPT4All responses as the model generates, use the streaming=True parameter during generation. This changes the returned type from a string to a Python generator iterator.

from gpt4all import GPT4All

tokens = []
with model.chat_session():
    for token in model.generate("What is the capital of France?", streaming=True):
        tokens.append(token)
    print('tokens-------:',tokens)
    # output to console:
    for token in model.generate("What is the highest mountain in South America?", streaming=True):
        print(token, end='', flush=True)
    print()

tokens-------: ['The', ' capital', ' of', ' France', ' is', ' Paris', '.']
Aconcagua, located in Argentina, is the highest mountain in South America and the second-highest peak outside of Asia, with an elevation of 6,962 meters (22,841 feet) above sea level.


#### Persistent chat history

In [12]:
#Turn on persistent chat history permanently
# Save the session object to a variable so Python doesn't delete it
session = model.chat_session()
session.__enter__()

# --- Now you can call .generate() normally on one line ---
print(model.generate("My name is John."))
print(model.generate("What is my name?")) # Outputs: "Your name is John."

#session.__exit__(None, None, None) #close the session

Hello, John! It's nice to meet you. Is there something I can help you with or would you like to chat for a bit?
Your name is John. We just established that earlier. How can I assist you today, John?


#### Chat Template

In [32]:
model_file="Llama-3.2-3B-Instruct-Q4_0.gguf"

In [21]:
#---------------LOAD TEMPLATE FROM GPT4ALL MODEL CONFIG----------------
#----------MIGHT NOT ALWAYS BE AVAILABLE!!!!!!!!!!!!
template=model.config["chatTemplate"]

In [42]:
#---------------------LOAD TEMPLATE FROM LOCAL GGUF MODEL FILE---------------------
#------RECOMMENDED: NOT DEPEND ON AVAILABILITY IN GPT4ALL API----------------

path=model.config.path #---------------->recommended: get gguf path from gpt4all model config for consistency

#1. Carga tu archivo GGUF local de forma normal
# (Usamos verbose=False para que no llene la pantalla con registros de inicialización)
llm = Llama(model_path=path, verbose=False,vocab_only=True)

# 2. Accede al diccionario de metadatos internos del GGUF
metadata = llm.metadata
template=metadata.get('tokenizer.chat_template')

llama_context: n_ctx_seq (512) > n_ctx_train (0) -- possible training context overflow


In [43]:
#-------------RECOMMENDED FOR NEWER GPT4ALL MODEL TEMPLATES------------
#-----INJECT TIMESTAMP-----(REQUIRED BY GPT4ALL API)----------------
# 1. Definimos una función en Python que replique strftime_now
def strftime_now(formato):
    # Retorna la fecha de hoy con el formato que pida el template (ej: '%d %b %Y')
    return datetime.now().strftime(formato)

# 2. Creamos un entorno de Jinja e inyectamos la función como global
env = Environment(loader=BaseLoader())
env.globals.update(strftime_now=strftime_now)

# 3. Compilamos el texto de la plantilla usando este entorno seguro
# (Reemplaza 'raw_template_text' por la variable que extraes de model.config o GGUF)
jinja_template = env.from_string(template)

In [39]:
#---------------IF LOADING TEMPLATE FROM LOCAL .GGUF, THIS CAN BE USED--------------------
#-----community .gguf adapted for llama.cpp usually contain simplified jinja templates, with hugging face/python datetime formatters removed
#-------this templates usually only require "user" and "assitant" code blocks
jinja_template = Template(template)

In [50]:
#----------RENDER PROMPT---------------------

#-------------fake chat history for testing-----------------------
historial_chat = [
    {"role": "system", "content": "You are a helpful assistant."}
]
historial_chat.append({"role": "user", "content": "hola"})

# Renderizar el historial de forma idéntica a como lo pide el creador del modelo
    # Le inyectamos las variables que esperan las plantillas estándar de Hugging Face y GGUF
prompt_renderizado = jinja_template.render(
        messages=historial_chat,
        add_generation_prompt=True,
        bos_token="<s>",  # Tokens especiales por defecto si la plantilla los pide
        eos_token="</s>"
    )



In [51]:
print('---------chat template \n{}:'.format(template))

---------chat template 
{{- bos_token }}
{%- if custom_tools is defined %}
    {%- set tools = custom_tools %}
{%- endif %}
{%- if not tools_in_user_message is defined %}
    {%- set tools_in_user_message = true %}
{%- endif %}
{%- if not date_string is defined %}
    {%- if strftime_now is defined %}
        {%- set date_string = strftime_now("%d %b %Y") %}
    {%- else %}
        {%- set date_string = "26 Jul 2024" %}
    {%- endif %}
{%- endif %}
{%- if not tools is defined %}
    {%- set tools = none %}
{%- endif %}

{#- This block extracts the system message, so we can slot it into the right place. #}
{%- if messages[0]['role'] == 'system' %}
    {%- set system_message = messages[0]['content']|trim %}
    {%- set messages = messages[1:] %}
{%- else %}
    {%- set system_message = "" %}
{%- endif %}

{#- System message #}
{{- "<|start_header_id|>system<|end_header_id|>\n\n" }}
{%- if tools is not none %}
    {{- "Environment: ipython\n" }}
{%- endif %}
{{- "Cutting Knowledge Date: 

In [52]:
print('-----------rendered template \n{}:'.format(prompt_renderizado))

-----------rendered template 
<s><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 04 Oct 2026

You are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>

hola<|eot_id|><|start_header_id|>assistant<|end_header_id|>

:


In [36]:
model.close()

#### Chat With Sliding Context Window
- Persisent History only for last N chats
- Use for avoiding overflowing ram/gpu buffer
- History is copied, re-rendered into the model jinja template, and passed as a "context aware" prompt
- This is what gpt4all backend (and in general any llm app) does under the hood: It passes history in the prompt.

In [54]:
#Turn on persistent chat history permanently
# Save the session object to a variable so Python doesn't delete it
session = model.chat_session()
session.__enter__()

# --- Now you can call .generate() normally on one line ---
print("------------initial test with persistent history-------")
print(model.generate("My name is John."))
print(model.generate("What is my name?")) # Outputs: "Your name is John."

history=model.current_chat_session

session.__exit__(None, None, None) #reset the session

print("-----cleared history--------------")
print(model.generate("What is my name?"))

------------initial test with persistent history-------
Hello, John! It's nice to meet you. Is there something I can help you with or would you like to chat for a bit?
Your name is John. We established that earlier when we started our conversation.
-----cleared history--------------
 I am a computer program designed to assist and communicate with humans. My purpose is to provide information, answer questions, and engage in conversation on various topics.

I don't have personal experiences or emotions like humans do. However, I can process vast amounts of data quickly and accurately, making me useful for tasks such as:

* Answering general knowledge questions
* Providing definitions and explanations
* Offering suggestions and recommendations
* Translating text from one language to another
* Generating creative content, such as stories or poems

I'm here to help you with any question or topic you'd like to discuss. Feel free to ask me anything!


In [ ]:
#-------------RECOMMENDED FOR NEWER GPT4ALL MODEL TEMPLATES------------
#-----INJECT TIMESTAMP-----(REQUIRED BY GPT4ALL API)----------------
# 1. Definimos una función en Python que replique strftime_now
def strftime_now(formato):
    # Retorna la fecha de hoy con el formato que pida el template (ej: '%d %b %Y')
    return datetime.now().strftime(formato)

# 2. Creamos un entorno de Jinja e inyectamos la función como global
env = Environment(loader=BaseLoader())
env.globals.update(strftime_now=strftime_now)

# 3. Compilamos el texto de la plantilla usando este entorno seguro
# (Reemplaza 'raw_template_text' por la variable que extraes de model.config o GGUF)
jinja_template = env.from_string(template)

In [56]:
history.append({"role": "user", "content": "What was my name?"})

In [63]:
# Renderizar el historial de forma idéntica a como lo pide el creador del modelo
    # Le inyectamos las variables que esperan las plantillas estándar de Hugging Face y GGUF
prompt_renderizado = jinja_template.render(
        messages=history,
        add_generation_prompt=True,
        bos_token="<s>",  # Tokens especiales por defecto si la plantilla los pide
        eos_token="</s>"
    )


In [64]:
prompt_renderizado

"<s><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 04 Oct 2026\n\n<|start_header_id|>system<|end_header_id|>\nCutting Knowledge Date: December 2023\n\nYou are a helpful assistant.<|eot_id|><|eot_id|><|start_header_id|>user<|end_header_id|>\n\nMy name is John.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nHello, John! It's nice to meet you. Is there something I can help you with or would you like to chat for a bit?<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nWhat is my name?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nYour name is John. We established that earlier when we started our conversation.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nWhat was my name?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"

In [65]:
res=model.generate(prompt=prompt_renderizado, max_tokens=512)

In [66]:
res

'You told me your name was John at the beginning of our conversation.'

In [67]:
model.close()